# Exercise 3: Data Preparation

A model learns only from the table it is given, so the table is prepared first: the gaps are filled, the outliers are understood, the categories become numbers, and the scales are made comparable. Every preparation step learns something from the data, such as a median, a list of categories, or a scale, and it learns it from the training rows alone. The result is a table ready for modelling in which every decision has a reason that the data support.

The worked example is the Automobile data in `data/automobile.csv`: 205 cars imported into the United States in 1985, with 25 columns that describe each car and its list price. The table is small and messy. It marks a missing value with `?`, mixes numbers, words, and labels, and holds a few cars that are far larger and more expensive than the rest. The hourly bike rentals in `data/bike-hours.csv` from Exercise 2 return for one section, because filling a gap in time needs a series.

The analysis follows three steps:

- find the missing values, explain why they are missing, and decide for each column whether to drop or to fill,
- detect the outliers, encode the categories, and bring the columns to comparable scales, and
- join every step in one pipeline, show what leaks when a step sees the test rows, and record each decision with its reason.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, MissingIndicator, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer, RobustScaler,
                                   StandardScaler)

DATA = Path("data")
SEED = 2026
rng = np.random.default_rng(SEED)

In [ ]:
# The course style. The colours repeat template/themes/data-science.css, so a change there is repeated here.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.facecolor": GROUND,
    "figure.constrained_layout.use": True,
    "savefig.facecolor": GROUND,
    "savefig.transparent": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Archivo", "DejaVu Sans"],
    "font.size": 12,
    "text.color": INK,
    "axes.facecolor": GROUND,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "grid.color": GRID,
    "xtick.color": INK,
    "ytick.color": INK,
    "legend.frameon": False,
    "image.cmap": "cividis",
})
sns.set_palette(PALETTE)

## 1. The Table

The file marks a missing value with a question mark, so `read_csv` takes `na_values="?"`. Without it, every column that holds a `?` would be read as text.

In [ ]:
cars = pd.read_csv(DATA / "automobile.csv", na_values="?")
cars.shape

In [ ]:
cars.head()

Each row is one model of car. The response is `price` in US dollars. The other columns are of four kinds:

- numerical measurements, such as `curb-weight` in pounds, `engine-size` in cubic inches, and `horsepower`,
- counts written as words, `num-of-doors` and `num-of-cylinders`, whose categories have an order,
- labels without an order, such as `body-style`, `fuel-type`, and `drive-wheels`, and
- `make`, a label with 22 values, several of which occur only once or twice.

`symboling` is an insurance risk rating from -3 (safe) to 3 (risky), and `normalized-losses` is the average insurance loss payment per insured vehicle year. The makes `alfa-romero` and `peugot` are spelled as the data spell them.

In [ ]:
cars.describe().T.round(1)

## 2. Discovering Missing Values

Missing values are counted per column and drawn row by row, because the pattern of the gaps says more than their number.

In [ ]:
missing = cars.isna().sum()
missing[missing > 0]

In [ ]:
gaps = cars.columns[cars.isna().any()]
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.imshow(cars[gaps].isna().T, aspect="auto", interpolation="none", cmap="Greys")
ax.set(yticks=range(len(gaps)), yticklabels=gaps, xlabel="Row of the table", title="Missing values, one dark mark per gap")
ax.grid(False)
plt.show()

The gaps are of three kinds, and each calls for its own decision:

- `price` is the response, and four cars lack it.
- `bore` and `stroke` are missing for the same four cars. They are the Mazda cars with a rotary engine, which has no cylinders and therefore no bore and no stroke. The value does not exist rather than being unrecorded.
- `normalized-losses` is missing for 41 cars, and the share depends on the make.

In [ ]:
cars.loc[cars["bore"].isna(), ["make", "engine-type", "num-of-cylinders", "bore", "stroke"]]

In [ ]:
by_make = cars["normalized-losses"].isna().groupby(cars["make"]).mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(by_make.index, by_make, color=PALETTE[0])
ax.set(ylabel="Share missing", title="Missing normalized-losses by make")
ax.tick_params(axis="x", rotation=60)
plt.show()

## 3. Mechanisms of Missingness

Three mechanisms describe why a value is missing:

- **Missing completely at random (MCAR).** The chance of a gap depends on nothing, so the observed values are a fair sample of all values.
- **Missing at random (MAR).** The chance of a gap depends on other observed columns, here the make, so a model that uses those columns can fill it.
- **Missing not at random (MNAR).** The chance of a gap depends on the missing value itself, so no observed column fully explains it.

The losses of `normalized-losses` are missing for every car of four makes and for none of seven others, so they are not missing completely at random. Whether the gap also depends on the losses themselves cannot be checked from the table. A simulation shows what each mechanism does to the mean of the observed values: the horsepower of the complete rows is hidden three ways, for 30 % of the cars each time.

In [ ]:
complete = cars.dropna(subset=["horsepower"])
hp, size = complete["horsepower"].to_numpy(), complete["engine-size"].to_numpy()
hidden = {
    "MCAR": rng.random(len(hp)) < 0.3,
    "MAR, by engine size": size > np.quantile(size, 0.7),
    "MNAR, by horsepower": hp > np.quantile(hp, 0.7),
}
pd.Series({name: hp[~mask].mean() for name, mask in hidden.items()} | {"all values": hp.mean()}).round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, (name, mask) in zip(axes, hidden.items()):
    ax.scatter(size[~mask], hp[~mask], s=14, label="observed")
    ax.scatter(size[mask], hp[mask], s=14, marker="x", label="hidden")
    ax.set(title=f"{name}: mean {hp[~mask].mean():.1f}", xlabel="Engine size")
axes[0].set_ylabel("Horsepower")
axes[0].legend(loc="upper left")
plt.show()

Under MCAR the observed mean stays close to the true one. Under MAR and MNAR the powerful cars are the ones that go missing, and the observed mean falls. The difference is that under MAR the engine size, which is observed, predicts the missing horsepower, while under MNAR nothing observed does.

## 4. Deletion or Imputation

Deleting is the simplest way to remove a gap, and the counts show its cost. `dropna()` drops every row with a gap, and `dropna(axis=1)` drops every column with one.

In [ ]:
complete_rows = cars.dropna()
complete_columns = cars.dropna(axis=1)
lost_makes = sorted(set(cars["make"]) - set(complete_rows["make"]))
len(complete_rows), lost_makes, list(cars.columns.difference(complete_columns.columns))

Dropping the rows keeps 159 of the 205 cars, and four makes disappear entirely, because their losses are always missing. The remaining cars are no longer a fair sample of the makes. Dropping the columns removes seven of them, the response among them. Neither is acceptable for the whole table, so each column is decided on its own:

- a row without the response is dropped, because a car without a price can neither train nor test a model of the price, and
- every other gap is filled, and the fill is learned from the training rows.

In [ ]:
cars = cars.dropna(subset=["price"]).reset_index(drop=True)
len(cars)

## 5. The Split Comes First

Every later step learns something from the data, so the test rows are set aside before any of them runs. `train_test_split` shuffles the rows and holds out a quarter of them, and `random_state` fixes the shuffle so that the split is the same on every run.

In [ ]:
X, y = cars.drop(columns="price"), cars["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)
len(X_train), len(X_test)

From here on, every decision is made on `X_train` and `y_train`. The test rows are only transformed with what the training rows taught, and Section 12 shows what goes wrong otherwise.

## 6. Imputation

`SimpleImputer` fills each column with one value learned from the training rows: the mean, the median, the most frequent value, or a constant.

In [ ]:
numeric = ["symboling", "normalized-losses", "wheel-base", "length", "width", "height", "curb-weight", "engine-size",
           "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm", "city-mpg", "highway-mpg"]
median = SimpleImputer(strategy="median").fit(X_train[numeric])
pd.Series(median.statistics_, index=numeric).round(2)

`KNNImputer` fills a gap with the mean of the nearest complete rows, and `IterativeImputer` predicts each column from the others with a regression model, in rounds. Both use the relationships between columns. To compare them, a fifth of the known horsepower values of the training rows are hidden, imputed, and compared with the truth by the mean absolute difference.

In [ ]:
known = X_train[numeric].dropna(subset=["horsepower"])
mask = np.random.default_rng(SEED).random(len(known)) < 0.2
masked = known.copy()
masked.loc[mask, "horsepower"] = np.nan
column = numeric.index("horsepower")
truth = known.loc[mask, "horsepower"].to_numpy()


def impute_scaled(imputer: KNNImputer, frame: pd.DataFrame) -> np.ndarray:
    # Distances need comparable units, so the neighbours are found on standardized columns.
    scaler = StandardScaler().fit(frame)
    return scaler.inverse_transform(imputer.fit_transform(scaler.transform(frame)))


filled = {
    "mean": SimpleImputer(strategy="mean").fit_transform(masked),
    "median": SimpleImputer(strategy="median").fit_transform(masked),
    "k nearest neighbours": impute_scaled(KNNImputer(n_neighbors=5), masked),
    "iterative": IterativeImputer(max_iter=20, random_state=SEED).fit_transform(masked),
}
errors = pd.Series({name: np.abs(truth - values[mask, column]).mean() for name, values in filled.items()})
errors.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, name in zip(axes, ["median", "iterative"]):
    ax.scatter(truth, filled[name][mask, column], color=PALETTE[0])
    ax.axline((100, 100), slope=1, color=INK, linewidth=1)
    ax.set(title=f"{name}, off by {errors[name]:.1f} on average", xlabel="True horsepower")
axes[0].set_ylabel("Imputed horsepower")
plt.show()

The median gives every hidden car the same horsepower, so its error is as large as the spread of the column. The model-based imputers use the engine size and the weight, which predict horsepower well, and cut the error to about a third.

A filled value hides that it was ever missing, and the absence itself can carry information. `MissingIndicator` adds one column of flags per column that had gaps during fitting, and `add_indicator=True` makes an imputer append them.

In [ ]:
MissingIndicator().fit(X_train[numeric]).get_feature_names_out()

The training rows have gaps in three numerical columns, so three flags are added. The flag of `bore` marks exactly the rotary engines, for which an imputed bore is a stand-in for a value that does not exist. The test rows also miss `horsepower` and `peak-rpm`, which the training rows never miss. The imputer still fills them, but no flag is created for them, because a flag is learned only from the training rows.

## 7. Interpolation

In a series, a gap has neighbours in time, and interpolation fills it from them. The hourly bike rentals of Exercise 2 have no row for 165 of the 17,544 hours of the two years. Reindexing to every hour with `asfreq("h")` makes the gaps visible.

In [ ]:
bikes = pd.read_csv(DATA / "bike-hours.csv")
hours = pd.to_datetime(bikes["date"]) + pd.to_timedelta(bikes["hour"], unit="h")
rentals = pd.Series(bikes["rentals"].to_numpy(), index=hours).asfreq("h")
gap = rentals.isna()
gap.sum(), gap[gap].index.hour.isin(range(2, 6)).sum()

Most gaps fall between 2 and 6 a.m., where the neighbouring hours count only a few rentals, which is consistent with hours in which nobody rented a bike and so no row was written. The longest gap covers 36 hours on 29 and 30 October 2012, when the storm Sandy shut the system down. Neither kind of gap is a measurement that failed at random.

In [ ]:
storm = slice("2012-10-28", "2012-10-31")
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(rentals.interpolate(method="time")[storm], color=PALETTE[1], linestyle="--", label="interpolate(method='time')")
ax.plot(rentals.interpolate(method="time", limit=3, limit_area="inside")[storm], color=PALETTE[0],
        label="interpolate(limit=3)")
ax.plot(rentals[storm], color=INK, label="observed")
ax.set(ylabel="Rentals per hour", title="Interpolation across the storm")
ax.legend()
plt.show()

Linear interpolation draws a straight line across the 36 hours and invents rentals for a closed system. `limit=3` fills only the first three hours of a gap and leaves the rest missing, which suits the short night gaps and leaves the storm visible. For the night gaps, `fillna(0)` is the other reasonable choice, because a missing row there most likely means no rentals.

## 8. Outliers

An outlier is a value far from the rest. The IQR rule flags a value more than 1.5 interquartile ranges beyond the quartiles, and the z-score flags a value more than three standard deviations from the mean.

In [ ]:
def iqr_outliers(values: pd.Series) -> pd.Series:
    q1, q3 = values.quantile([0.25, 0.75])
    return (values < q1 - 1.5 * (q3 - q1)) | (values > q3 + 1.5 * (q3 - q1))


ratio = X_train["compression-ratio"]
flagged = iqr_outliers(ratio)
z_flagged = np.abs(stats.zscore(ratio)) > 3
flagged.sum(), z_flagged.sum(), pd.crosstab(flagged, X_train["fuel-type"])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
sns.stripplot(x=ratio, y=X_train["fuel-type"], hue=flagged, palette=[PALETTE[0], PALETTE[1]], ax=ax)
ax.set(xlabel="Compression ratio", ylabel="", title="Cars flagged by the IQR rule")
ax.legend(title="flagged")
plt.show()

The IQR rule flags every diesel car, because a diesel engine compresses its air about twice as much as a petrol engine. These values are correct, and `fuel-type` already explains them, so they are kept. The z-score flags none of them, because the diesel cars inflate the standard deviation that it divides by. The quartiles of this column are close together, so the rule also flags a few petrol cars that are only slightly unusual.

The two rules look at one column at a time. `IsolationForest` isolates a row with random splits and flags the rows that are isolated quickly, and `LocalOutlierFactor` flags a row whose neighbourhood is much sparser than that of its neighbours. Both look at several columns together and return -1 for an outlier.

In [ ]:
pair = X_train[["engine-size", "curb-weight"]]
forest = IsolationForest(contamination=0.05, random_state=SEED).fit_predict(pair) == -1
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit_predict(pair) == -1
X_train.loc[forest | lof, ["make", "engine-size", "curb-weight"]].assign(forest=forest[forest | lof], lof=lof[forest | lof])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, flags, name in [(axes[0], forest, "Isolation Forest"), (axes[1], lof, "Local Outlier Factor")]:
    ax.scatter(pair["engine-size"], pair["curb-weight"], c=np.where(flags, PALETTE[1], PALETTE[0]))
    ax.set(title=f"{name}, {flags.sum()} flagged", xlabel="Engine size")
axes[0].set_ylabel("Curb weight")
plt.show()

Both flag the largest engines, which belong to the most expensive makes, and the lightest car, and the factor also flags a few cars that sit apart from a dense group. These are real cars, and their prices are part of what the data describe, so they stay. An outlier is removed only when it is an error, and otherwise it is kept, capped, or transformed.

Capping replaces every value beyond a bound with the bound. The bounds are quantiles of the training rows, and the same bounds are applied to the test rows.

In [ ]:
low, high = X_train["engine-size"].quantile([0.01, 0.99])
capped = X_test["engine-size"].clip(low, high)
(low, high), (X_test["engine-size"] != capped).sum()

Three test cars have a larger engine than 99 % of the training cars, and capping would record them as if their engine were 246 cubic inches. Capping suits a value that is believed to be extreme by error or that would dominate a fit. The large engines here are correct, so the table keeps them.

## 9. Encoding

A model needs numbers, so every label becomes one or more numerical columns, and the encoding follows the kind of variable.

- An **ordinal** variable has an order, so `OrdinalEncoder` maps its categories to 0, 1, 2, and so on, in an order passed explicitly.
- A **nominal** variable has no order, so `OneHotEncoder` gives each category its own 0 or 1 column.
- A variable with **many categories**, several of them rare, would add one column per category. Frequency encoding replaces each category with its share of the training rows instead.

In [ ]:
ordinal = ["num-of-doors", "num-of-cylinders"]
orders = [["two", "four"], ["two", "three", "four", "five", "six", "eight", "twelve"]]
encoded = OrdinalEncoder(categories=orders).fit_transform(X_train[ordinal].dropna())
pd.DataFrame(encoded, columns=ordinal).value_counts().sort_index()

Without `categories=`, the encoder sorts the words alphabetically and puts `eight` before `four`. The explicit order also holds `twelve`, which only a test car has, because the order of the counts is known before any data are seen. The codes keep the order but not the distances: `eight` becomes 5 and `twelve` becomes 6.

In [ ]:
nominal = ["fuel-type", "aspiration", "body-style", "drive-wheels", "engine-location", "engine-type", "fuel-system"]
one_hot = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False).fit(X_train[nominal])
len(one_hot.get_feature_names_out()), one_hot.get_feature_names_out()[:6]

`drop="if_binary"` keeps a single column for a variable with two categories, such as `fuel-type`, because its second column would repeat the first. `handle_unknown="ignore"` encodes a category the training rows never held as a row of zeros instead of raising an error.

Scikit-learn has no frequency encoder, so a small transformer learns the shares in `fit` and maps them in `transform`. A make that the training rows never held gets the share 0.

In [ ]:
class FrequencyEncoder(TransformerMixin, BaseEstimator):
    def fit(self, X: pd.DataFrame, y: object = None) -> "FrequencyEncoder":
        self.shares_ = {column: X[column].value_counts(normalize=True) for column in X.columns}
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        return np.column_stack([X[column].map(share).fillna(0).to_numpy(dtype=float)
                                for column, share in self.shares_.items()])

    def get_feature_names_out(self, input_features: object = None) -> np.ndarray:
        return np.array([f"{column}_frequency" for column in self.shares_])


frequency = FrequencyEncoder().fit(X_train[["make"]])
frequency.shares_["make"].head().round(3), sorted(set(X_test["make"]) - set(X_train["make"]))

The make `renault` occurs only among the test cars, so it is encoded as 0, the share it has in the training rows.

## 10. Scaling and Power Transforms

Distances and penalties treat every column alike, so columns measured in pounds and in inches are brought to comparable scales. Each scaler learns its statistics from the training rows.

- `StandardScaler` subtracts the mean and divides by the standard deviation, which suits most columns.
- `MinMaxScaler` maps the training range to [0, 1], which suits a bounded column, but one extreme value squeezes all the others.
- `RobustScaler` subtracts the median and divides by the interquartile range, which keeps the bulk of a column with outliers on a usable scale.

In [ ]:
column = X_train[["compression-ratio"]]
scaled = {"raw": column.to_numpy(), "standard": StandardScaler().fit_transform(column),
          "min-max": MinMaxScaler().fit_transform(column), "robust": RobustScaler().fit_transform(column)}
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
for ax, (name, values) in zip(axes, scaled.items()):
    ax.hist(values, bins=30, color=PALETTE[0])
    ax.set(title=name, yticks=[])
plt.show()
pd.DataFrame({name: [values.min(), np.median(values), values.max()] for name, values in scaled.items()},
             index=["min", "median", "max"]).round(2)

The compression ratio holds the petrol cars near 9 and the diesel cars near 22. Min-max scaling squeezes all petrol cars into the bottom fifth of [0, 1], the standard scaler lets the diesel cars pull the mean and the spread, and the robust scaler spreads the petrol cars around 0 and places the diesel cars far out. Every scaler moves and stretches a column but keeps its shape.

`PowerTransformer` changes the shape: it finds the power that makes a column closest to normal. Box-Cox needs positive values, and Yeo-Johnson also accepts zero and negative values. The price is skewed to the right.

In [ ]:
box_cox = PowerTransformer(method="box-cox").fit(y_train.to_frame())
transformed = box_cox.transform(y_train.to_frame()).ravel()
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(y_train, bins=25, color=PALETTE[0])
axes[0].set(title=f"Price, skew {stats.skew(y_train):.2f}", xlabel="US dollars", yticks=[])
axes[1].hist(transformed, bins=25, color=PALETTE[2])
axes[1].set(title=f"Box-Cox, lambda {box_cox.lambdas_[0]:.2f}, skew {stats.skew(transformed):.2f}", yticks=[])
plt.show()

Box-Cox raises the price to the power $\lambda$, with $\lambda = 0$ meaning the logarithm, so the fitted $\lambda$ lies between the logarithm and the reciprocal. The price is the response, so the prepared table keeps it in dollars, and whether a model predicts the price or its transform is decided with the model.

## 11. One Pipeline

`ColumnTransformer` applies one chain of steps to each group of columns, and `make_pipeline` chains the steps of a group. The whole preparation becomes one object whose `fit` learns every statistic from the rows it is given and whose `transform` applies them to any rows. `set_output(transform="pandas")` makes it return a data frame with named columns.

In [ ]:
numeric_steps = make_pipeline(IterativeImputer(max_iter=20, random_state=SEED, add_indicator=True), StandardScaler())
ordinal_steps = make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder(categories=orders), StandardScaler())
preparation = ColumnTransformer([
    ("numeric", numeric_steps, numeric),
    ("ordinal", ordinal_steps, ordinal),
    ("nominal", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), nominal),
    ("make", make_pipeline(FrequencyEncoder(), StandardScaler()), ["make"]),
]).set_output(transform="pandas")
prepared_train = preparation.fit_transform(X_train)
prepared_test = preparation.transform(X_test)
prepared_train.shape, prepared_test.shape, prepared_train.isna().sum().sum(), prepared_test.isna().sum().sum()

In [ ]:
prepared_train.iloc[:5, [0, 1, 7, 15, 16, 17, 20, 45]].round(2)

The 25 columns become 46 numbers with no gaps. Each name starts with the group that produced it, such as `numeric__` or `nominal__`, and the fitted steps can be read back from the pipeline.

In [ ]:
fitted_numeric = preparation.named_transformers_["numeric"]
pd.Series(fitted_numeric[-1].mean_, index=fitted_numeric[0].get_feature_names_out()).round(2).tail(5)

The last three means are the shares of the training cars with each flag set, because a flag is 0 or 1 before it is scaled.

## 12. Leakage

Leakage happens when information from the rows a result is judged on enters the fitting. The test rows stand in for cars the preparation has never seen, so nothing about them may shape it. The same preparation is fitted twice, once on all rows and once on the training rows only, and the learned values are compared.

In [ ]:
leaky = clone(preparation).fit(X)
honest = clone(preparation).fit(X_train)


def learned(fitted: ColumnTransformer) -> pd.Series:
    scaler = fitted.named_transformers_["numeric"][-1]
    shares = fitted.named_transformers_["make"][0].shares_["make"]
    engine = numeric.index("engine-size")
    return pd.Series({"mean of engine-size": scaler.mean_[engine],
                      "standard deviation of engine-size": scaler.scale_[engine],
                      "share of renault": shares.get("renault", 0.0)})


pd.DataFrame({"all rows": learned(leaky), "training rows": learned(honest)}).round(3)

Every learned value moves. The preparation fitted on all rows knows the make `renault`, which only the test cars have, and its scale of the engine size is stretched by the large test engines. The capping bounds of Section 8 move the same way.

In [ ]:
bounds = pd.Series({"all rows": X["engine-size"].quantile(0.99), "training rows": X_train["engine-size"].quantile(0.99)})
bounds.round(1), [(X_test["engine-size"] > bound).sum() for bound in bounds]

The leaky effect is that the test rows look more ordinary than new cars would. The prepared test columns are compared by their standard deviation, which would be 1 for rows like the training rows.

In [ ]:
measured = ["numeric__engine-size", "numeric__curb-weight", "numeric__horsepower"]
spread = pd.DataFrame({"all rows": leaky.transform(X_test)[measured].std(),
                       "training rows": honest.transform(X_test)[measured].std()})
spread.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(X_train["engine-size"], bins=25, color=PALETTE[0], label="training cars")
axes[0].hist(X_test["engine-size"], bins=25, color=PALETTE[1], alpha=0.7, label="test cars")
for (name, bound), style in zip(bounds.items(), ["--", "-"]):
    axes[0].axvline(bound, color=INK, linestyle=style, label=f"99th percentile, {name}")
axes[0].set(xlabel="Engine size", yticks=[])
axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.02), ncols=2)
spread.plot.barh(ax=axes[1], color=[PALETTE[1], PALETTE[0]])
axes[1].axvline(1, color=INK, linewidth=1)
axes[1].set(xlabel="Standard deviation of the scaled test rows")
axes[1].invert_yaxis()
axes[1].legend(title="fitted on", loc="lower left", bbox_to_anchor=(0, 1.02), ncols=2)
plt.show()

Fitted on the training rows, the preparation shows the test cars as they are: more spread out than the training cars, with three engines beyond the training 99th percentile and a make it has never seen. Fitted on all rows, the test cars helped set the scale and the bounds, so they look closer to typical, and any result measured on them would be judged on easier rows than new cars are.

A pipeline prevents this because it is one object fitted on whatever rows it is given. It is fitted on the training rows and only transforms the test rows, and any procedure that later splits the training rows again refits the whole preparation on each part, so held-out rows never shape it.

## 13. The Prepared Table

The prepared training and test tables have the same 46 columns, no gaps, and every statistic learned from the training rows. A scaled training column has mean 0 and standard deviation 1 by construction. A scaled test column departs from both, and the departure is how new cars differ from the training cars.

In [ ]:
prepared_test.agg(["mean", "std"]).T.iloc[[1, 7, 11, 15, 16, 45]].round(2)

Every decision in the table is recorded in the pipeline, with a reason that the data support:

| Columns | Decision | Reason |
| --- | --- | --- |
| `price` | Rows without it are dropped. | A car without the response can neither train nor test a model of it. |
| `normalized-losses` | Imputed from the other columns and flagged. | The gaps depend on the make, so they are not missing completely at random, and the flag keeps the absence visible. |
| `bore`, `stroke` | Imputed and flagged. | A rotary engine has no bore and no stroke, so the flag marks the cars whose value does not exist. |
| `horsepower`, `peak-rpm` | Imputed from the other columns. | The iterative imputer misses a hidden horsepower by about a third of what the median misses. |
| `num-of-doors`, `num-of-cylinders` | Filled with the most frequent value and encoded in their order. | The counts have an order, which an alphabetical encoding would break. |
| The nominal labels | One-hot encoded, with one column for a binary label and zeros for an unseen category. | The labels have no order, so each category gets its own column, and the seven labels need only 25 of them. |
| `make` | Frequency encoded. | Twenty-two makes, several with one or two cars, would add a column for each, and an unseen make gets the share 0. |
| The diesel compression ratios and the large engines | Kept. | They are correct values, explained by the fuel type and by the size of the car. |
| Every column | Standardized. | Columns in pounds, inches, and shares become comparable for distances and penalties. |
| Every learned value | Learned from the training rows inside one pipeline. | Fitted on all rows, the preparation makes the test cars look more ordinary than new cars. |